# Archived research notebook

Original filename: `preprocessing fifth基于伪影检测，剔除电源线影响and高通滤波0.2HZ.ipynb`  
Purpose: Apply high-pass filtering and line-noise mitigation.  
Historical research version retained for traceability. The original test data are unavailable; legacy outputs, execution state, and attachments were removed for public release.


In [ ]:
print(f"MNE 版本: {mne.__version__}")

In [ ]:
#这份代码也是对数据没有影响的 你可以和下面的代码进行对比 会发现跑出来同一份数据这份代码的某些图会很混乱 而下面那份代码跑出来的图是规整的
import os
import mne
import matplotlib.pyplot as plt


# 定义要处理的文件夹路径
folder_path = r"data/chb-mit/chb01"

# 定义要去除的电源线干扰频率
notch_freqs = [60, 120]

# 遍历文件夹中的所有文件
for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.endswith('_raw_no_proj.fif'):
            # 构建文件的完整路径
            fif_path = os.path.join(root, file)
            try:
                # 读取 fif 文件
                raw = mne.io.read_raw_fif(fif_path, preload=True)

                # 绘制滤波前的功率谱密度图
                fig1 = raw.compute_psd().plot()
                fig1.suptitle(f"Before Notch Filtering - {file}")

                # 陷波滤波，去除 60Hz 和 120Hz 干扰
                raw.notch_filter(freqs=notch_freqs)

                # 绘制滤波后的功率谱密度图
                fig2 = raw.compute_psd().plot()
                fig2.suptitle(f"After Notch Filtering - {file}")

                # 显示图形
                plt.show()

                print(f"文件 {file} 陷波滤波处理完成，可视化展示已完成")
            except Exception as e:
                print(f"处理 {file} 时出错: {e}")

print("所有文件处理完成")
    

In [ ]:
#这串代码其实是对数据没有影响的 因为我没有保存数据 我只是想让你看看0.1HZ和0.2HZ的差别在哪里
import mne
import numpy as np
import matplotlib.pyplot as plt

# 文件地址
file_path = r"data/chb-mit/chb01/chb01_01_raw_no_proj.fif"

# 读取原始数据
raw = mne.io.read_raw_fif(file_path, preload=True)

# 不同的截止频率
cutoffs = (0.1, 0.2)

# 定义频段
bands = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta': (13, 30),
    'gamma': (30, 100),
    'hfo': (80, 128)  # 调整高频振荡频段上限以符合采样频率要求
}

# 初始化结果字典
results = {}

for cutoff in cutoffs:
    # 复制数据并进行高通滤波
    raw_highpass = raw.copy().filter(l_freq=cutoff, h_freq=None)

    # 频域特征分析 - 功率谱密度
    spectrum = raw_highpass.compute_psd(fmin=0, fmax=128)
    psd = spectrum.get_data()
    freqs = spectrum.freqs

    band_powers = {}
    for band, (fmin, fmax) in bands.items():
        idx_band = np.logical_and(freqs >= fmin, freqs <= fmax)
        band_power = np.mean(psd[:, idx_band], axis=1)
        band_powers[band] = band_power

    # 统计特征分析
    data = raw_highpass.get_data()
    mean_values = np.mean(data, axis=1)
    std_values = np.std(data, axis=1)
    var_values = np.var(data, axis=1)

    # 相关性分析
    correlation_matrix = np.corrcoef(data)

    results[cutoff] = {
        'band_powers': band_powers,
        'mean': mean_values,
        'std': std_values,
        'var': var_values,
        'correlation_matrix': correlation_matrix
    }

# 可视化频域特征 - 各频段功率分布
for band in bands.keys():
    plt.figure()
    for i, cutoff in enumerate(cutoffs):
        if cutoff == 0.1:
            plt.plot(results[cutoff]['band_powers'][band], 
                     label=f'{cutoff} Hz', 
                     color='black',  # 用黑色
                     linewidth=3,    # 更大线宽
                     alpha=1)        # 不透明
        else:
            plt.plot(results[cutoff]['band_powers'][band], 
                     label=f'{cutoff} Hz', 
                     color='skyblue', # 浅蓝色
                     linewidth=2,    # 稍小线宽
                     alpha=0.7)      # 有透明度
    plt.title(f'{band} band power distribution')
    plt.xlabel('Channel')
    plt.ylabel('Power')
    plt.legend()
    plt.show()

# 可视化统计特征 - 均值、标准差和方差
for stat in ['mean', 'std', 'var']:
    plt.figure()
    for cutoff in cutoffs:
        plt.plot(results[cutoff][stat], label=f'{cutoff} Hz')
    plt.title(f'{stat.capitalize()} values across channels')
    plt.xlabel('Channel')
    plt.ylabel(stat.capitalize())
    plt.legend()
    plt.show()

# 可视化相关性分析结果
for cutoff in cutoffs:
    plt.figure()
    plt.imshow(results[cutoff]['correlation_matrix'], cmap='viridis', interpolation='nearest')
    plt.title(f'Correlation matrix for {cutoff} Hz cutoff')
    plt.colorbar()
    plt.show()

In [ ]:
import mne
import os
import matplotlib.pyplot as plt

# 文件夹路径
folder_path = r"data/chb-mit/chb01"

# 获取文件夹下所有以 _raw_no_proj.fif 结尾的文件
fif_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith("_raw_no_proj.fif")]

# 截止频率
highpass_cutoff = 0.2


def add_arrows(axes, powerline_freqs):
    # 标记陷波频率位置
    for ax in axes:
        for freq in powerline_freqs:
            ax.axvline(x=freq, color='r', linestyle='--')


for file_path in fif_files:
    try:
        # 读取原始数据
        raw = mne.io.read_raw_fif(file_path, preload=True)

        # 动态获取采样频率
        sfreq = raw.info['sfreq']

        # 生成电源线干扰频率及其谐波，最大不超过采样频率的一半
        powerline_freqs = tuple([i * 60 for i in range(1, int(sfreq / 120) + 1)])

        # 选择所有数据通道
        all_picks = mne.pick_types(raw.info, eeg=True, meg=False, exclude='bads')

        # 进行高通滤波
        raw_highpass = raw.copy().filter(l_freq=highpass_cutoff, h_freq=None, picks=all_picks)

        # 进行陷波滤波
        raw_notch = raw_highpass.copy().notch_filter(freqs=powerline_freqs, picks=all_picks)

        # 绘制滤波前后的功率谱密度图
        for title, data in zip(["Un", "Notch "], [raw_highpass, raw_notch]):
            fig = data.compute_psd(fmax=sfreq / 2).plot(
                average=True, amplitude=False, picks="data", exclude="bads"
            )
            fig.suptitle(f"{title}filtered {os.path.basename(file_path)}", size="xx-large", weight="bold")
            add_arrows(fig.axes[:2], powerline_freqs)
            plt.show()

    except Exception as e:
        print(f"处理文件 {file_path} 时出现错误: {e}")
    

In [ ]:
#这份才是这个处理真正有用的代码
import mne
import os
import matplotlib.pyplot as plt

# 文件夹路径
folder_path = r"data/chb-mit/chb01"

# 获取文件夹下所有以 _raw_no_proj.fif 结尾的文件
fif_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith("_raw_no_proj.fif")]

# 截止频率
highpass_cutoff = 0.2


def add_arrows(axes, powerline_freqs):
    # 标记陷波频率位置
    for ax in axes:
        for freq in powerline_freqs:
            ax.axvline(x=freq, color='r', linestyle='--')


for file_path in fif_files:
    try:
        # 读取原始数据
        raw = mne.io.read_raw_fif(file_path, preload=True)

        # 动态获取采样频率
        sfreq = raw.info['sfreq']

        # 生成电源线干扰频率及其谐波，最大不超过采样频率的一半
        powerline_freqs = tuple([i * 60 for i in range(1, int(sfreq / 120) + 1)])

        # 选择所有数据通道
        all_picks = mne.pick_types(raw.info, eeg=True, meg=False, exclude='bads')

        # 进行高通滤波
        raw_highpass = raw.copy().filter(l_freq=highpass_cutoff, h_freq=None, picks=all_picks)

        # 
        raw_notch = raw_highpass.copy().notch_filter(freqs=powerline_freqs, picks=all_picks)

        # 绘制滤波前后的功率谱密度图
        for title, data in zip(["Un", "Notch "], [raw_highpass, raw_notch]):
            fig = data.compute_psd(fmax=sfreq / 2).plot(
                average=True, amplitude=False, picks="data", exclude="bads"
            )
            fig.suptitle(f"{title}filtered {os.path.basename(file_path)}", size="xx-large", weight="bold")
            add_arrows(fig.axes[:2], powerline_freqs)
            plt.show()

        # 保存处理后的数据为FIF文件
        output_file_path = file_path.replace("_raw_no_proj.fif", "_processed.fif")
        raw_notch.save(output_file_path, overwrite=True)
        print(f"处理后的数据已保存到 {output_file_path}")

    except Exception as e:
        print(f"处理文件 {file_path} 时出现错误: {e}")
    